## Iteration Changelog
- Added Welch t-tests for card/stack/board-risk features across win/loss groups.
- Added stage model artifact diagnostics and quick scenario checks.
- Open issues: calibrate decision thresholds with backtests by tournament segment.

In [ ]:
from pathlib import Path

import joblib
import pandas as pd
from scipy import stats

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATA_PATH = ROOT / "data" / "cleanedGambling.csv"
MODEL_PATH = ROOT / "poker_models.pkl"
FEATURE_PATH = ROOT / "feature_names.pkl"

df = pd.read_csv(DATA_PATH)
models = joblib.load(MODEL_PATH)
stage_features = joblib.load(FEATURE_PATH)

print(df.shape)
print(list(models.keys()))

## Welch T-Tests (Win vs Lose)

We test whether key engineered features differ significantly across win/loss outcomes.

In [ ]:
test_features = [
    "preflop_preflop_strength",
    "flop_hand_strength",
    "turn_hand_strength",
    "river_hand_strength",
    "preflop_hero_stack_bb",
    "flop_spr",
    "turn_spr",
    "river_spr",
    "flop_board_shared_strength_risk",
    "turn_board_shared_strength_risk",
    "river_board_shared_strength_risk",
]

test_features = [c for c in test_features if c in df.columns]
rows = []
for c in test_features:
    a = pd.to_numeric(df.loc[df["won_flag"] == 1, c], errors="coerce").dropna()
    b = pd.to_numeric(df.loc[df["won_flag"] == 0, c], errors="coerce").dropna()
    if len(a) < 3 or len(b) < 3:
        continue
    t_stat, p = stats.ttest_ind(a, b, equal_var=False)
    rows.append({
        "feature": c,
        "mean_win": a.mean(),
        "mean_lose": b.mean(),
        "t_stat": t_stat,
        "p_value": p,
        "significant": p < 0.05,
    })

ttest_df = pd.DataFrame(rows).sort_values("p_value")
ttest_df.head(20)

In [ ]:
# Stage feature contract and quick model diagnostics
for stage, feats in stage_features.items():
    print(f"\n{stage} ({len(feats)} features)")
    print(feats[:12])

print("\nModel objects loaded:", list(models.keys()))

In [ ]:
# Quick scenario check: safe board vs scary shared board
from scripts.features.poker_hand_strength import build_stage_feature_payload
from scripts.models.stage_win_predictor import predict_stage_win_probability

safe = build_stage_feature_payload(
    "turn",
    ["Ah", "Kd"],
    ["Qs", "7d", "2c", "9h"],
    total_bet=50,
    current_pot=90,
    hero_stack=110,
    table_stacks=[110, 85, 140],
    big_blind=2,
)
scary = build_stage_feature_payload(
    "turn",
    ["As", "Kd"],
    ["Qh", "Jh", "2h", "9h"],
    total_bet=50,
    current_pot=90,
    hero_stack=110,
    table_stacks=[110, 85, 140],
    big_blind=2,
)

safe_p = predict_stage_win_probability("turn", safe)
scary_p = predict_stage_win_probability("turn", scary)

print("safe probability:", round(safe_p, 4), "risk:", round(safe["board_shared_strength_risk"], 3))
print("scary probability:", round(scary_p, 4), "risk:", round(scary["board_shared_strength_risk"], 3))